In [ ]:
!pip install -q datasets transformers accelerate evaluate jiwer soundfile librosa
!pip install -q optimum[onnxruntime] onnx onnxruntime

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA available: True
GPU: Tesla T4


In [ ]:
from datasets import load_dataset

fleurs_test = load_dataset("google/fleurs", "ta_in", split="test")

print(fleurs_test)
print(fleurs_test[0].keys())
print("Transcript:", fleurs_test[0]["transcription"])

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


parquet-data/ta_in/train-00000-of-00001.(…):   0%|          | 0.00/1.99G [00:00<?, ?B/s]

parquet-data/ta_in/validation-00000-of-0(…):   0%|          | 0.00/288M [00:00<?, ?B/s]

parquet-data/ta_in/test-00000-of-00001.p(…):   0%|          | 0.00/489M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2367 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/377 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/591 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
    num_rows: 591
})
dict_keys(['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'])
Transcript: இது வேதியியல் ph என அழைக்கப்படுகிறது நீங்கள் சிவப்பு முட்டைக்கோஸ் சாற்றைப் பயன்படுத்தி ஒரு குறிகாட்டியை உருவாக்கலாம்


In [ ]:
import torch
import evaluate
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from tqdm.auto import tqdm

model_id = "openai/whisper-tiny"
processor = WhisperProcessor.from_pretrained(model_id, language="tamil", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(model_id).to("cuda")
model.eval()

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

predictions = []
references = []

N_EVAL = 200  # start small; increase later

for i in tqdm(range(N_EVAL)):
    sample = fleurs_test[i]
    audio = sample["audio"]

    inputs = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        generated_ids = model.generate(inputs.input_features, language="tamil", task="transcribe")

    pred = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    ref = sample["transcription"]

    predictions.append(pred)
    references.append(ref)

wer = 100 * wer_metric.compute(predictions=predictions, references=references)
cer = 100 * cer_metric.compute(predictions=predictions, references=references)

print(f"Baseline Whisper-tiny WER: {wer:.2f}%")
print(f"Baseline Whisper-tiny CER: {cer:.2f}%")

preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/151M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

  0%|          | 0/200 [00:00<?, ?it/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Baseline Whisper-tiny WER: 193.20%
Baseline Whisper-tiny CER: 157.35%


In [ ]:
from pathlib import Path
import gc
import json
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

ROOT = Path("/content/tamil_whisper")
ROOT.mkdir(parents=True, exist_ok=True)

BASE_MODEL = "openai/whisper-tiny"
FINETUNED_DIR = ROOT / "finetuned"
EXPORT_HF_DIR = ROOT / "selected-hf"
ONNX_DIR = ROOT / "onnx"

N_TRAIN = 1000
N_VALID = 100
N_TEST = min(200, len(fleurs_test))

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
print("Experiment directory:", ROOT)

if device.type != "cuda":
    raise RuntimeError(
        "Enable a GPU runtime and fix the CUDA-enabled PyTorch "
        "installation before fine-tuning."
    )

# Release the model loaded in Cell 4.
if "model" in globals():
    del model

gc.collect()
torch.cuda.empty_cache()

Device: cuda
Experiment directory: /content/tamil_whisper


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pathlib import Path
import gc
import json
import random
import numpy as np
import torch
matches = list(
    Path("/content/drive/MyDrive").rglob(
        "common_voice_tamil_subset.zip"
    )
)

for i, path in enumerate(matches):
    print(f"[{i}] {path}")

if not matches:
    raise FileNotFoundError(
        "Subset ZIP not found. In Colab's Files panel, locate "
        "the ZIP under drive/MyDrive, right-click it, and "
        "choose 'Copy path'."
    )

[0] /content/drive/MyDrive/common_voice_tamil_subset.zip


In [ ]:
assert len(matches) == 1, (
    "Multiple ZIPs found. Set ZIP_PATH to the correct one."
)

ZIP_PATH = matches[0]
print("Using:", ZIP_PATH)

Using: /content/drive/MyDrive/common_voice_tamil_subset.zip


In [ ]:
import zipfile

EXTRACT_DIR = Path("/content/cv_tamil_subset")
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

assert ZIP_PATH.is_file(), f"ZIP not found: {ZIP_PATH}"

with zipfile.ZipFile(ZIP_PATH) as archive:
    archive.extractall(EXTRACT_DIR)

CV_ROOT = EXTRACT_DIR / "ta"

assert (CV_ROOT / "train.tsv").is_file(), "train.tsv missing"
assert (CV_ROOT / "dev.tsv").is_file(), "dev.tsv missing"
assert (CV_ROOT / "clips").is_dir(), "clips/ missing"

print("Dataset extracted successfully.")
print("CV_ROOT =", CV_ROOT)

Dataset extracted successfully.
CV_ROOT = /content/cv_tamil_subset/ta


In [ ]:
import pandas as pd

train_preview = pd.read_csv(
    CV_ROOT / "train.tsv",
    sep="\t",
    dtype=str,
    keep_default_na=False,
)

valid_preview = pd.read_csv(
    CV_ROOT / "dev.tsv",
    sep="\t",
    dtype=str,
    keep_default_na=False,
)

for name, df in [
    ("Training", train_preview),
    ("Validation", valid_preview),
]:
    missing = [
        filename
        for filename in df["path"]
        if not (CV_ROOT / "clips" / filename).is_file()
    ]

    print(f"{name} rows: {len(df)}")
    print(f"{name} missing audio files: {len(missing)}")

    assert not missing, (
        f"{name}: missing audio files, for example {missing[:3]}"
    )

display(train_preview[["path", "sentence"]].head())

Training rows: 1000
Training missing audio files: 0
Validation rows: 100
Validation missing audio files: 0


,path,sentence
0,common_voice_ta_27259257.mp3,"அவன் அரசவை இருந்து, அவனுக்கு அறம் உரைத்து வந்த..."
1,common_voice_ta_25220810.mp3,கோயில் வாயிலிலே நீண்டு உயர்ந்த கோபுரம் இருக்காது.
2,common_voice_ta_30013284.mp3,நூலறிவுக்கு ஆக்கம் சேர்ப்பது ஆய்வியல் சார்ந்த ...
3,common_voice_ta_25883923.mp3,இந்த திருத்தங்களில் பெரும்பாலானவை சிவில் தொழிற...
4,common_voice_ta_25947258.mp3,"அதிர்ஷ்டத்தின் ஒரு கட்டத்தில், Kadriansன் திட்..."


In [ ]:
from IPython.display import Audio, display
import librosa

example_path = (
    CV_ROOT / "clips" / train_preview.iloc[0]["path"]
)

waveform, sr = librosa.load(
    example_path,
    sr=16000,
    mono=True,
)

print("Transcript:", train_preview.iloc[0]["sentence"])
display(Audio(waveform, rate=sr))

Transcript: அவன் அரசவை இருந்து, அவனுக்கு அறம் உரைத்து வந்தார் பலராவர்.


In [ ]:
from pathlib import Path
import gc

import librosa
import numpy as np
import pandas as pd

from datasets import (
    Dataset,
    Features,
    Array2D,
    Sequence,
    Value,
)
from transformers import WhisperProcessor


CV_ROOT = Path("/content/cv_tamil_subset/ta")

CACHE_DIR = Path("/content/tamil_feature_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

BASE_MODEL = "openai/whisper-tiny"

# The training cache-recovery cell must have created this.
assert "train_data" in globals(), (
    "train_data is missing. Run the training cache-recovery "
    "cell first, then run this cell."
)
assert len(train_data) > 0, "The recovered training dataset is empty."

processor = WhisperProcessor.from_pretrained(
    BASE_MODEL,
    language="tamil",
    task="transcribe",
)


def load_split(filename):
    df = pd.read_csv(
        CV_ROOT / filename,
        sep="\t",
        dtype=str,
        keep_default_na=False,
    )

    df = df[
        df["path"].str.strip().ne("")
        & df["sentence"].str.strip().ne("")
    ].copy()

    df["audio_path"] = df["path"].map(
        lambda name: str(CV_ROOT / "clips" / name)
    )

    exists = df["audio_path"].map(
        lambda path: Path(path).is_file()
    )

    if not exists.all():
        missing = df.loc[
            ~exists, "audio_path"
        ].head().tolist()

        raise FileNotFoundError(
            f"Missing audio files: {missing}"
        )

    return Dataset.from_pandas(
        df[["audio_path", "sentence"]],
        preserve_index=False,
    )


# Keep both raw splits available for later evaluation.
# This does not recompute training features.
cv_train = load_split("train.tsv")
cv_valid = load_split("dev.tsv")

feature_schema = Features({
    "input_features": Array2D(
        shape=(80, 3000),
        dtype="float32",
    ),
    "labels": Sequence(Value("int64")),
    "keep": Value("bool"),
})


def prepare_example(example):
    waveform, sr = librosa.load(
        example["audio_path"],
        sr=16000,
        mono=True,
        dtype=np.float32,
    )

    labels = processor.tokenizer(
        example["sentence"]
    ).input_ids

    duration = len(waveform) / sr

    keep = (
        0 < duration <= 30
        and 0 < len(labels) <= 448
    )

    if keep:
        features = processor.feature_extractor(
            waveform,
            sampling_rate=sr,
            return_tensors="np",
        ).input_features[0]

        features = np.asarray(
            features,
            dtype=np.float32,
        )
    else:
        features = np.zeros(
            (80, 3000),
            dtype=np.float32,
        )
        labels = []

    return {
        "input_features": features,
        "labels": labels,
        "keep": keep,
    }


def prepare_split(dataset, name):
    prepared = dataset.map(
        prepare_example,
        batched=False,
        num_proc=1,
        remove_columns=dataset.column_names,
        features=feature_schema,
        writer_batch_size=4,
        keep_in_memory=False,
        cache_file_name=str(
            CACHE_DIR / f"{name}_features.arrow"
        ),
        load_from_cache_file=False,
        desc=f"Preparing {name}",
    )

    filtered = prepared.filter(
        lambda keep: keep,
        input_columns=["keep"],
        batched=False,
        num_proc=1,
        writer_batch_size=4,
        keep_in_memory=False,
        cache_file_name=str(
            CACHE_DIR / f"{name}_filtered_fixed.arrow"
        ),
        load_from_cache_file=False,
        desc=f"Filtering {name}",
    )

    return filtered.remove_columns(["keep"])


gc.collect()

# Reuse recovered train_data; prepare validation only.
valid_data = prepare_split(cv_valid, "valid")

gc.collect()

assert len(train_data) > 0
assert len(valid_data) > 0

print("Usable training clips:", len(train_data))
print("Usable validation clips:", len(valid_data))
print("Feature cache:", CACHE_DIR)
print("Ready for the collator, metrics, and training cells.")

AssertionError: train_data is missing. Run the training cache-recovery cell first, then run this cell.

In [ ]:
from pathlib import Path
from datasets import Dataset

CACHE_DIR = Path("/content/tamil_feature_cache")
train_cache = CACHE_DIR / "train_features.arrow"

assert train_cache.is_file(), (
    "Training feature cache is missing. "
    "Rerun preprocessing with the corrected filter."
)

prepared_train = Dataset.from_file(str(train_cache))

train_data = prepared_train.filter(
    lambda keep: keep,
    input_columns=["keep"],
    batched=False,
    num_proc=1,
    writer_batch_size=4,
    keep_in_memory=False,
    cache_file_name=str(
        CACHE_DIR / "train_filtered_fixed.arrow"
    ),
    load_from_cache_file=False,
    desc="Filtering training clips",
).remove_columns(["keep"])

print("Usable training clips:", len(train_data))
assert len(train_data) > 0

AssertionError: Training feature cache is missing. Rerun preprocessing with the corrected filter.

Training Whisper-tiny

In [ ]:
from pathlib import Path
import gc
import numpy as np
import torch

from transformers import (
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    set_seed,
)

assert "train_data" in globals(), "Training data is missing."
assert "valid_data" in globals(), "Validation data is missing."
assert len(train_data) > 0
assert len(valid_data) > 0

assert torch.cuda.is_available(), (
    "CUDA is unavailable. Fix the GPU runtime/PyTorch "
    "installation before training."
)

print("GPU:", torch.cuda.get_device_name(0))
print("Training clips:", len(train_data))
print("Validation clips:", len(valid_data))

set_seed(42)

# Release an earlier model if one is still loaded.
if "trainer" in globals():
    del trainer

if "model" in globals():
    del model

gc.collect()
torch.cuda.empty_cache()

BASE_MODEL = "openai/whisper-tiny"

model = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL
)

model.generation_config.language = "tamil"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
model.generation_config.do_sample = False
model.generation_config.num_beams = 1

# Convert only retrieved examples to tensors—not the whole dataset.
train_data = train_data.with_format(
    "torch",
    columns=["input_features", "labels"],
)

valid_data = valid_data.with_format(
    "torch",
    columns=["input_features", "labels"],
)

decoder_start_token_id = model.config.decoder_start_token_id

In [ ]:
def data_collator(features):
    input_features = torch.stack([
        torch.as_tensor(
            item["input_features"],
            dtype=torch.float32,
        )
        for item in features
    ])

    label_batch = processor.tokenizer.pad(
        [
            {
                "input_ids": torch.as_tensor(
                    item["labels"]
                ).tolist()
            }
            for item in features
        ],
        return_tensors="pt",
    )

    labels = label_batch["input_ids"].masked_fill(
        label_batch["attention_mask"].ne(1),
        -100,
    )

    if (
        labels[:, 0] == decoder_start_token_id
    ).all().item():
        labels = labels[:, 1:]

    return {
        "input_features": input_features,
        "labels": labels,
    }


# Check one small batch before starting training.
test_batch = data_collator([
    train_data[0],
    train_data[1] if len(train_data) > 1 else train_data[0],
])

print("Audio features:", test_batch["input_features"].shape)
print("Labels:", test_batch["labels"].shape)

del test_batch

In [ ]:
import evaluate

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")


def compute_metrics(pred):
    prediction_ids = pred.predictions

    if isinstance(prediction_ids, tuple):
        prediction_ids = prediction_ids[0]

    prediction_ids = np.where(
        prediction_ids >= 0,
        prediction_ids,
        processor.tokenizer.pad_token_id,
    )

    label_ids = np.where(
        pred.label_ids != -100,
        pred.label_ids,
        processor.tokenizer.pad_token_id,
    )

    predictions = processor.batch_decode(
        prediction_ids,
        skip_special_tokens=True,
    )

    references = processor.batch_decode(
        label_ids,
        skip_special_tokens=True,
    )

    return {
        "wer": 100 * wer_metric.compute(
            predictions=predictions,
            references=references,
        ),
        "cer": 100 * cer_metric.compute(
            predictions=predictions,
            references=references,
        ),
    }

In [ ]:
assert Path("/content/drive/MyDrive").is_dir(), (
    "Mount Google Drive before continuing."
)

RUN_DIR = Path(
    "/content/drive/MyDrive/tamil_whisper_experiment"
)

FINETUNED_DIR = RUN_DIR / "finetuned"
RUN_DIR.mkdir(parents=True, exist_ok=True)

training_args = Seq2SeqTrainingArguments(
    output_dir=str(RUN_DIR / "checkpoints"),

    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    per_device_eval_batch_size=1,

    learning_rate=1e-5,
    warmup_ratio=0.1,
    num_train_epochs=3,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,

    predict_with_generate=True,
    generation_max_length=448,
    generation_num_beams=1,
    eval_accumulation_steps=1,

    fp16=True,

    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    save_total_limit=2,

    remove_unused_columns=False,
    label_names=["labels"],

    dataloader_num_workers=0,
    dataloader_pin_memory=False,

    report_to=[],
    seed=42,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=valid_data,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
)

In [ ]:
train_result = trainer.train()

In [ ]:
trainer.save_model(str(FINETUNED_DIR))
processor.save_pretrained(str(FINETUNED_DIR))

trainer.save_metrics(
    "train",
    train_result.metrics,
)

trainer.save_state()

print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation WER:", trainer.state.best_metric)
print("Saved fine-tuned model:", FINETUNED_DIR)

Comparison of fine tuned model vs original Whisper-tiny

In [ ]:
from pathlib import Path
import gc
import json

import numpy as np
import pandas as pd
import torch

RUN_DIR = Path(
    "/content/drive/MyDrive/tamil_whisper_experiment"
)
FINETUNED_DIR = RUN_DIR / "finetuned"
EVAL_DIR = RUN_DIR / "fleurs_evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

assert (FINETUNED_DIR / "config.json").is_file(), (
    f"Saved model not found at {FINETUNED_DIR}. "
    "Check the save location printed after training."
)

# Release training objects before loading evaluation models.
if "trainer" in globals():
    del trainer

if "model" in globals():
    del model

gc.collect()
torch.cuda.empty_cache()

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Evaluation device:", device)
print("Fine-tuned model:", FINETUNED_DIR)

Evaluation device: cuda
Fine-tuned model: /content/drive/MyDrive/tamil_whisper_experiment/finetuned


In [ ]:
from datasets import load_dataset

if "fleurs_test" not in globals():
    fleurs_test = load_dataset(
        "google/fleurs",
        "ta_in",
        split="test",
    )

# Same first 200 samples as the original experiment.
# For a full-test evaluation, change this to len(fleurs_test).
N_TEST = min(200, len(fleurs_test))

test_indices = list(range(N_TEST))

print("Available FLEURS test examples:", len(fleurs_test))
print("Examples selected:", N_TEST)

with (EVAL_DIR / "test_indices.json").open("w") as f:
    json.dump(test_indices, f)

Available FLEURS test examples: 591
Examples selected: 200


In [ ]:
import librosa
import jiwer

from tqdm.auto import tqdm
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
)


def evaluate_on_fleurs(model_path, name):
    eval_processor = WhisperProcessor.from_pretrained(
        model_path,
        language="tamil",
        task="transcribe",
    )

    eval_model = (
        WhisperForConditionalGeneration.from_pretrained(
            model_path
        )
        .to(device)
        .eval()
    )

    eval_model.generation_config.language = "tamil"
    eval_model.generation_config.task = "transcribe"
    eval_model.generation_config.forced_decoder_ids = None

    # Whisper's timestamp processor requires a scalar EOS token ID.
    eos_id = eval_processor.tokenizer.eos_token_id

    if not isinstance(eos_id, int):
        raise TypeError(
            f"Expected an integer EOS token ID, got {eos_id!r}"
        )

    eval_model.generation_config.eos_token_id = eos_id

    print(
        f"{name}: using eos_token_id={eos_id} "
        f"({type(eos_id).__name__})"
    )

    rows = []

    try:
        for index in tqdm(test_indices, desc=name):
            sample = fleurs_test[index]
            audio = sample["audio"]

            waveform = np.asarray(
                audio["array"],
                dtype=np.float32,
            )
            sr = audio["sampling_rate"]

            if sr != 16000:
                waveform = librosa.resample(
                    waveform,
                    orig_sr=sr,
                    target_sr=16000,
                )
                sr = 16000

            duration = len(waveform) / sr
            is_long = duration > 30.0

            inputs = eval_processor(
                waveform,
                sampling_rate=sr,
                return_tensors="pt",
                return_attention_mask=True,
                truncation=False,
                padding="longest" if is_long else "max_length",
            ).to(device)

            with torch.inference_mode():
                generated_ids = eval_model.generate(
                    input_features=inputs.input_features,
                    attention_mask=inputs.attention_mask,
                    language="tamil",
                    task="transcribe",
                    num_beams=1,
                    do_sample=False,
                    max_new_tokens=440,
                    return_timestamps=is_long,
                    eos_token_id=eos_id,
                )

            prediction = eval_processor.batch_decode(
                generated_ids,
                skip_special_tokens=True,
            )[0]

            rows.append({
                "index": index,
                "duration_seconds": duration,
                "reference": sample["transcription"],
                "prediction": prediction,
            })

            # Save partial progress every 20 examples.
            if len(rows) % 20 == 0:
                pd.DataFrame(rows).to_csv(
                    EVAL_DIR / f"{name}_predictions.csv",
                    index=False,
                )

        output = pd.DataFrame(rows)

        references = output["reference"].tolist()
        predictions = output["prediction"].tolist()

        metrics = {
            "model": name,
            "samples": len(output),
            "wer": 100 * jiwer.wer(
                references,
                predictions,
            ),
            "cer": 100 * jiwer.cer(
                references,
                predictions,
            ),
        }

        output.to_csv(
            EVAL_DIR / f"{name}_predictions.csv",
            index=False,
        )

        with (EVAL_DIR / f"{name}_metrics.json").open("w") as f:
            json.dump(metrics, f, indent=2)

        return metrics, output

    finally:
        del eval_model
        gc.collect()
        torch.cuda.empty_cache()

In [ ]:
baseline_metrics, baseline_output = evaluate_on_fleurs(
    "openai/whisper-tiny",
    "baseline",
)

print(
    f"Baseline WER: {baseline_metrics['wer']:.2f}%"
)
print(
    f"Baseline CER: {baseline_metrics['cer']:.2f}%"
)

baseline: using eos_token_id=50257 (int)


baseline:   0%|          | 0/200 [00:00<?, ?it/s]

Baseline WER: 195.66%
Baseline CER: 159.23%


In [ ]:
finetuned_metrics, finetuned_output = evaluate_on_fleurs(
    str(FINETUNED_DIR),
    "finetuned",
)

print(
    f"Fine-tuned WER: {finetuned_metrics['wer']:.2f}%"
)
print(
    f"Fine-tuned CER: {finetuned_metrics['cer']:.2f}%"
)

finetuned: using eos_token_id=50257 (int)


finetuned:   0%|          | 0/200 [00:00<?, ?it/s]

`generation_config` default values have been modified to match model-specific defaults: {'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50358, 50359, 50360, 50361, 50362], 'begin_suppress_tokens': [220, 50257]}. If this is not desired, please set these values explicitly.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTok

Fine-tuned WER: 103.07%
Fine-tuned CER: 46.49%


In [ ]:
comparison = pd.DataFrame([
    baseline_metrics,
    finetuned_metrics,
]).set_index("model")

display(comparison.round(2))

comparison.to_csv(
    EVAL_DIR / "baseline_vs_finetuned.csv"
)

paired_predictions = baseline_output[
    ["index", "reference", "prediction"]
].rename(
    columns={"prediction": "baseline"}
).merge(
    finetuned_output[
        ["index", "prediction"]
    ].rename(
        columns={"prediction": "finetuned"}
    ),
    on="index",
    validate="one_to_one",
)

paired_predictions.to_csv(
    EVAL_DIR / "paired_predictions.csv",
    index=False,
)

wer_change = (
    baseline_metrics["wer"]
    - finetuned_metrics["wer"]
)

cer_change = (
    baseline_metrics["cer"]
    - finetuned_metrics["cer"]
)

print(
    "WER reduction, baseline minus fine-tuned: "
    f"{wer_change:.2f} percentage points"
)
print(
    "CER reduction, baseline minus fine-tuned: "
    f"{cer_change:.2f} percentage points"
)

print("Evaluation results saved to:", EVAL_DIR)

,samples,wer,cer
model,,,
baseline,200,195.66,159.23
finetuned,200,103.07,46.49


WER reduction, baseline minus fine-tuned: 92.59 percentage points
CER reduction, baseline minus fine-tuned: 112.74 percentage points
Evaluation results saved to: /content/drive/MyDrive/tamil_whisper_experiment/fleurs_evaluation


In [ ]:
import pandas as pd
from pathlib import Path

EVAL_DIR = Path(
    "/content/drive/MyDrive/tamil_whisper_experiment/"
    "fleurs_evaluation"
)

paired = pd.read_csv(
    EVAL_DIR / "paired_predictions.csv",
    keep_default_na=False,
)

display(
    paired[
        ["reference", "baseline", "finetuned"]
    ].head(10)
)

,reference,baseline,finetuned
0,இது வேதியியல் ph என அழைக்கப்படுகிறது நீங்கள் ச...,இது வேதியில் பீயாச் சென்று அலைக்குப் படியில் ...,இது வேதியில் பீயச் சென் அலைக்கப்படியில் எழுது...
1,லக்கா சிங் பஜனையை வழங்கினார் பாடகர் ராஜு கண்டெ...,அழுது வழுது வழுது வழுது வழுது வழுது வழுது வழு...,அல்லகாச் சிங்கும் பதினியுவலங்கினர் பாடகர் ராஜ...
2,நீங்கள் உங்கள் சொந்த கருத்தைத் தவிர அரசாங்கங்க...,நீங்கள் வந்து வந்து வந்து வந்து வந்து வந்து வ...,நீங்களும்பல் சொன்ற கருத்தைக்க விரா அரசாங்கள் ...
3,முக்கியமான மேடைகளில் இசை நிகழ்ச்சி முடிந்திருந...,"முக்கிய மானம் மேடிக்கலி, இதை நிக்கலித்தி முடி...",முக்கி மானம் எனக்கலி இசை நிகழ்ச்சி முடிந்திரி...
4,அந்தப் பாலம் செப்டெம்பர் 2017 இல் பிரேசிலிய சு...,"அந்த பாலம் சப்ப்பர் இரண்டாயர்த்திப் பதன்றி, ப...",அந்தப் பாலம் செப்டம்பர் ரண்டாயர்திப் பதின் அழ...
5,பேருந்துகள் நாள் முழுவதும் மாவட்டங்களுக்கு இடை...,பேரந்துகள் நான் முலுக்குத்து மாவட்டுங்களுக்கு...,பேருந்துகள் நால் முழுவது மாவட்டுங்களுக்கையிடை...
6,துணி மிகவும் வெப்பமடைதலை அனுமதிக்காதபடி கவனமாக...,தூனி மையவும் விப்பும் மடிந்ததலை அனும் மதிக்கப...,துணி மிகவும் விப்பு மடிந்ததலை அணுமதிக்கப்பல்த...
7,இதன் விளைவாக இரண்டு2 மீன் வகைகள் அழிந்துவிட்டன...,இதன் மிலை வாகக்கு இரண்டு மீன் வாகக்கைக்குக்கு...,இதன் மிலை வாக இயிரண்டு மீன் வகைக்கை அழிநிவிட்...
8,ரோம் நகரில் பல இடங்களில் ஏராளமான பெரிய தொலைக்க...,ஏதாவது தொலைக் காத்திற்கு தெரிக்கிற்கு அமைக்கு...,உரோம் நகரில் பல யரங்களில் யரால முனும் பெரிய த...
9,"""வலைப்பதிவுகளின் சரியான பயன்பாடு ""மாணவர்களை மே...",வளைப்பதிப்புக்குள்ளின் சரியானப் பயன்பாடும் மா...,வழைப்பதிபுகளின் சரியாணப் பயன்பாடு மானவர்களி ந...


Export

In [ ]:
%pip install -q transformers openai-whisper onnx onnxruntime sherpa-onnx soundfile

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 12.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 119.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 121.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 126.0 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

RUN_DIR = Path(
    "/content/drive/MyDrive/tamil_whisper_experiment"
)

HF_DIR = RUN_DIR / "finetuned"

# Work locally, then copy verified artifacts to Drive.
WORK_DIR = Path("/content/tamil_whisper_export")
ONNX_DIR = WORK_DIR / "onnx"

WORK_DIR.mkdir(parents=True, exist_ok=True)
ONNX_DIR.mkdir(parents=True, exist_ok=True)

assert (HF_DIR / "config.json").is_file(), (
    f"Saved checkpoint not found: {HF_DIR}"
)

weights = [
    *HF_DIR.glob("*.safetensors"),
    *HF_DIR.glob("pytorch_model*.bin"),
]

assert weights, "No checkpoint weights found."

print("Checkpoint:", HF_DIR)
print("Weights:", [p.name for p in weights])

Mounted at /content/drive
Checkpoint: /content/drive/MyDrive/tamil_whisper_experiment/finetuned
Weights: ['model.safetensors']


In [ ]:
# convert fine tuned weights
import re
import torch

from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor,
)

from whisper.model import (
    Whisper,
    ModelDimensions,
)

torch.set_num_threads(2)

hf_processor = WhisperProcessor.from_pretrained(
    str(HF_DIR),
    language="tamil",
    task="transcribe",
)

hf_model = (
    WhisperForConditionalGeneration.from_pretrained(
        str(HF_DIR),
        attn_implementation="eager",
    )
    .cpu()
    .float()
    .eval()
)

config = hf_model.config

assert config.d_model == 384
assert config.encoder_layers == 4
assert config.decoder_layers == 4
assert config.num_mel_bins == 80
assert config.vocab_size == 51865
assert len(hf_processor.tokenizer) == config.vocab_size

dims = ModelDimensions(
    n_mels=config.num_mel_bins,
    n_audio_ctx=config.max_source_positions,
    n_audio_state=config.d_model,
    n_audio_head=config.encoder_attention_heads,
    n_audio_layer=config.encoder_layers,
    n_vocab=config.vocab_size,
    n_text_ctx=config.max_target_positions,
    n_text_state=config.d_model,
    n_text_head=config.decoder_attention_heads,
    n_text_layer=config.decoder_layers,
)

openai_model = Whisper(dims).cpu().float().eval()


def convert_key(key):
    if key.startswith("model."):
        key = key[len("model."):]

    key = key.replace(
        "encoder.embed_positions.weight",
        "encoder.positional_embedding",
    )

    key = key.replace(
        "decoder.embed_positions.weight",
        "decoder.positional_embedding",
    )

    key = key.replace(
        "decoder.embed_tokens.weight",
        "decoder.token_embedding.weight",
    )

    key = re.sub(
        r"^(encoder|decoder)\.layers\.(\d+)\.",
        r"\1.blocks.\2.",
        key,
    )

    replacements = [
        ("self_attn_layer_norm.", "attn_ln."),
        ("encoder_attn_layer_norm.", "cross_attn_ln."),
        ("self_attn.q_proj.", "attn.query."),
        ("self_attn.k_proj.", "attn.key."),
        ("self_attn.v_proj.", "attn.value."),
        ("self_attn.out_proj.", "attn.out."),
        ("encoder_attn.q_proj.", "cross_attn.query."),
        ("encoder_attn.k_proj.", "cross_attn.key."),
        ("encoder_attn.v_proj.", "cross_attn.value."),
        ("encoder_attn.out_proj.", "cross_attn.out."),
        ("final_layer_norm.", "mlp_ln."),
        ("fc1.", "mlp.0."),
        ("fc2.", "mlp.2."),
    ]

    for old, new in replacements:
        key = key.replace(old, new)

    key = key.replace(
        "encoder.layer_norm.",
        "encoder.ln_post.",
    )

    key = key.replace(
        "decoder.layer_norm.",
        "decoder.ln.",
    )

    return key


hf_state = hf_model.state_dict()

# OpenAI Whisper uses the token embedding as its output projection.
assert torch.equal(
    hf_state["proj_out.weight"],
    hf_state["model.decoder.embed_tokens.weight"],
), "This conversion expects tied output-projection weights."

converted = {
    convert_key(key): tensor
    for key, tensor in hf_state.items()
    if key != "proj_out.weight"
}

expected = openai_model.state_dict()

assert set(converted) == set(expected), {
    "missing": sorted(set(expected) - set(converted)),
    "unexpected": sorted(set(converted) - set(expected)),
}

for key in expected:
    assert converted[key].shape == expected[key].shape, key

openai_model.load_state_dict(
    converted,
    strict=True,
)

print("All fine-tuned parameters converted and loaded.")

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

All fine-tuned parameters converted and loaded.


In [ ]:
from whisper.model import disable_sdpa

torch.manual_seed(42)

features = torch.randn(1, 80, 3000)

prefix_ids = [
    hf_model.config.decoder_start_token_id,
    *[
        token_id
        for _, token_id
        in hf_processor.get_decoder_prompt_ids(
            language="tamil",
            task="transcribe",
        )
    ],
]

decoder_ids = torch.tensor(
    [prefix_ids],
    dtype=torch.long,
)

with torch.inference_mode(), disable_sdpa():
    hf_logits = hf_model(
        input_features=features,
        decoder_input_ids=decoder_ids,
    ).logits

    openai_logits = openai_model(
        features,
        decoder_ids,
    )

difference = (
    hf_logits - openai_logits
).abs().max().item()

print("Maximum absolute logit difference:", difference)

torch.testing.assert_close(
    hf_logits,
    openai_logits,
    rtol=1e-3,
    atol=1e-3,
)

OPENAI_CHECKPOINT = WORK_DIR / "finetuned-openai.pt"

torch.save(
    {
        "dims": vars(dims),
        "model_state_dict": openai_model.state_dict(),
    },
    OPENAI_CHECKPOINT,
)

print("Conversion check passed.")
print("Converted checkpoint:", OPENAI_CHECKPOINT)

Maximum absolute logit difference: 8.392333984375e-05
Conversion check passed.
Converted checkpoint: /content/tamil_whisper_export/finetuned-openai.pt


In [ ]:
import gc

del hf_logits, openai_logits
del features, decoder_ids
del hf_state, converted, expected
del hf_model, openai_model

gc.collect()

1395

In [ ]:
# get sherpa exporter
import subprocess

SHERPA_REPO = WORK_DIR / "sherpa-onnx"

if not SHERPA_REPO.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/k2-fsa/sherpa-onnx.git",
            str(SHERPA_REPO),
        ],
        check=True,
    )

EXPORT_SCRIPT = (
    SHERPA_REPO
    / "scripts"
    / "whisper"
    / "export-onnx.py"
)

assert EXPORT_SCRIPT.is_file()

SHERPA_COMMIT = subprocess.check_output(
    [
        "git",
        "-C",
        str(SHERPA_REPO),
        "rev-parse",
        "HEAD",
    ],
    text=True,
).strip()

print("Exporter source commit:", SHERPA_COMMIT)

Exporter source commit: a63f82130d91d99a23a43f346104706881b31ff1


In [ ]:
# Export FP32 and INT8 ONNX models
import textwrap

RUNNER = WORK_DIR / "run_export.py"

RUNNER.write_text(
    textwrap.dedent(
        f"""
        import importlib.util
        import sys
        import torch
        import whisper
        from whisper.model import disable_sdpa

        torch.set_num_threads(1)
        torch.set_num_interop_threads(1)

        spec = importlib.util.spec_from_file_location(
            "sherpa_whisper_export",
            {str(EXPORT_SCRIPT)!r},
        )

        exporter = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(exporter)

        def load_finetuned_model(name):
            print("Exporting fine-tuned checkpoint:", {str(OPENAI_CHECKPOINT)!r})
            return whisper.load_model(
                {str(OPENAI_CHECKPOINT)!r},
                device="cpu",
            )

        exporter.load_model = load_finetuned_model

        # Use the export path expected by this upstream script.
        original_export = torch.onnx.export

        def legacy_export(*args, **kwargs):
            kwargs["dynamo"] = False
            return original_export(*args, **kwargs)

        torch.onnx.export = legacy_export

        sys.argv = ["export-onnx.py", "--model", "tiny"]

        with disable_sdpa():
            exporter.main()
        """
    ),
    encoding="utf-8",
)

subprocess.run(
    ["python", str(RUNNER)],
    cwd=str(ONNX_DIR),
    check=True,
)

print("FP32 export and INT8 quantization finished.")

FP32 export and INT8 quantization finished.


In [ ]:
import json
import shutil
import onnx
import onnxruntime as ort
import importlib.metadata

required_files = [
    "tiny-encoder.onnx",
    "tiny-decoder.onnx",
    "tiny-encoder.int8.onnx",
    "tiny-decoder.int8.onnx",
    "tiny-tokens.txt",
]

for filename in required_files:
    path = ONNX_DIR / filename
    assert path.is_file() and path.stat().st_size > 0, filename

session_options = ort.SessionOptions()
session_options.intra_op_num_threads = 1
session_options.inter_op_num_threads = 1

for filename in required_files:
    if not filename.endswith(".onnx"):
        continue

    path = ONNX_DIR / filename

    onnx.checker.check_model(str(path))

    session = ort.InferenceSession(
        str(path),
        sess_options=session_options,
        providers=["CPUExecutionProvider"],
    )

    print("Validated:", filename)
    del session

size_rows = []

for precision, suffix in [
    ("FP32", ""),
    ("INT8", ".int8"),
]:
    encoder = ONNX_DIR / f"tiny-encoder{suffix}.onnx"
    decoder = ONNX_DIR / f"tiny-decoder{suffix}.onnx"

    total = encoder.stat().st_size + decoder.stat().st_size

    size_rows.append({
        "precision": precision,
        "encoder_MiB": encoder.stat().st_size / 1024**2,
        "decoder_MiB": decoder.stat().st_size / 1024**2,
        "total_MiB": total / 1024**2,
    })

import pandas as pd

size_table = pd.DataFrame(size_rows)
display(size_table.round(2))

DRIVE_EXPORT_DIR = RUN_DIR / "sherpa_onnx"
DRIVE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

for filename in required_files:
    shutil.copy2(
        ONNX_DIR / filename,
        DRIVE_EXPORT_DIR / filename,
    )

# Keep the intermediate checkpoint for reproducibility.
shutil.copy2(
    OPENAI_CHECKPOINT,
    DRIVE_EXPORT_DIR / OPENAI_CHECKPOINT.name,
)

manifest = {
    "source_checkpoint": str(HF_DIR),
    "architecture": "multilingual-whisper-tiny",
    "language": "ta",
    "task": "transcribe",
    "sherpa_source_commit": SHERPA_COMMIT,
    "quantization": "dynamic QInt8 MatMul weights",
    "versions": {
        name: importlib.metadata.version(name)
        for name in [
            "torch",
            "transformers",
            "openai-whisper",
            "onnx",
            "onnxruntime",
            "sherpa-onnx",
        ]
    },
}

(DRIVE_EXPORT_DIR / "manifest.json").write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

size_table.to_csv(
    DRIVE_EXPORT_DIR / "model_sizes.csv",
    index=False,
)

print("Artifacts saved to Drive:", DRIVE_EXPORT_DIR)

Validated: tiny-encoder.onnx
Validated: tiny-decoder.onnx
Validated: tiny-encoder.int8.onnx
Validated: tiny-decoder.int8.onnx


,precision,encoder_MiB,decoder_MiB,total_MiB
0,FP32,35.89,109.19,145.08
1,INT8,12.33,85.68,98.01


Artifacts saved to Drive: /content/drive/MyDrive/tamil_whisper_experiment/sherpa_onnx


In [ ]:
from pathlib import Path
from google.colab import files

# Reuse the file you already uploaded.
existing_file = Path("/content/common_voice_ta_19083405.mp3")

if existing_file.is_file():
    TEST_WAV = existing_file
else:
    uploaded = files.upload()

    audio_files = [
        filename
        for filename in uploaded
        if Path(filename).suffix.lower() in {".wav", ".mp3"}
    ]

    assert len(audio_files) == 1, (
        "Upload exactly one Tamil audio file: MP3 or WAV."
    )

    TEST_WAV = Path(audio_files[0]).resolve()

    del uploaded

assert TEST_WAV.is_file(), f"Audio file not found: {TEST_WAV}"

print("Audio selected:", TEST_WAV)

Audio selected: /content/common_voice_ta_19083405.mp3


In [ ]:
import time
import librosa
import numpy as np
import sherpa_onnx

waveform, sr = librosa.load(
    TEST_WAV,
    sr=16000,
    mono=True,
    dtype=np.float32,
)

assert len(waveform) > 0

duration = len(waveform) / sr

assert duration <= 30, (
    "Use a recording under 30 seconds for this smoke test."
)

smoke_rows = []

for precision, suffix in [
    ("FP32", ""),
    ("INT8", ".int8"),
]:
    recognizer = sherpa_onnx.OfflineRecognizer.from_whisper(
        encoder=str(
            DRIVE_EXPORT_DIR / f"tiny-encoder{suffix}.onnx"
        ),
        decoder=str(
            DRIVE_EXPORT_DIR / f"tiny-decoder{suffix}.onnx"
        ),
        tokens=str(
            DRIVE_EXPORT_DIR / "tiny-tokens.txt"
        ),
        language="ta",
        task="transcribe",
        num_threads=2,
        provider="cpu",
        decoding_method="greedy_search",
    )

    stream = recognizer.create_stream()

    start = time.perf_counter()

    stream.accept_waveform(
        16000,
        np.ascontiguousarray(waveform),
    )
    recognizer.decode_stream(stream)

    elapsed = time.perf_counter() - start
    text = stream.result.text

    smoke_rows.append({
        "precision": precision,
        "audio_seconds": duration,
        "decode_seconds": elapsed,
        "text": text,
    })

    print(f"\n{precision}: {text}")
    print(f"Decode time: {elapsed:.2f} seconds")

    del stream, recognizer
    gc.collect()

pd.DataFrame(smoke_rows).to_csv(
    DRIVE_EXPORT_DIR / "smoke_test.csv",
    index=False,
)


FP32: முத்தும் அறிந்திடக் கூடுமன்றே அவ
Decode time: 1.14 seconds

INT8: உத்தும் அறிந்திடக் கூறுமன்றே அவள
Decode time: 0.96 seconds


### Why INT8?

INT8 is being tested to make the model smaller and faster for offline Android use. In this test, it decoded the clip in 0.96 seconds, compared with 1.14 seconds for FP32.

The outputs were slightly different, so faster decoding alone is not enough to choose INT8. **FP32** remains the reference model while WER/CER are compared. The final choice also depends on performance on the actual phone, since these timings were measured in Colab.

In [ ]:
from pathlib import Path
import json
import gc
import time

import librosa
import numpy as np
import pandas as pd
import sherpa_onnx

from datasets import load_dataset
from tqdm.auto import tqdm

RUN_DIR = Path(
    "/content/drive/MyDrive/tamil_whisper_experiment"
)

MODEL_DIR = RUN_DIR / "sherpa_onnx"
EVAL_DIR = RUN_DIR / "fleurs_evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

if "fleurs_test" not in globals():
    fleurs_test = load_dataset(
        "google/fleurs",
        "ta_in",
        split="test",
    )

# Reuse the exact test indices saved during HF evaluation.
indices_file = EVAL_DIR / "test_indices.json"

assert indices_file.is_file(), (
    "Saved evaluation indices are missing. Check EVAL_DIR."
)

test_indices = json.loads(indices_file.read_text())

print("Test recordings:", len(test_indices))

README.md:   0%|          | 0.00/386k [00:00<?, ?B/s]

parquet-data/ta_in/train-00000-of-00001.(…): reconstructing file:   0%|          |  0.00B / 1.99GB            

parquet-data/ta_in/train-00000-of-00001.(…): downloading bytes:           |  0.00B            

parquet-data/ta_in/validation-00000-of-0(…): reconstructing file:   0%|          |  0.00B /  288MB            

parquet-data/ta_in/validation-00000-of-0(…): downloading bytes:           |  0.00B            

...

parquet-data/ta_in/test-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2367 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/377 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/591 [00:00<?, ? examples/s]

Test recordings: 200


In [ ]:
%pip install -q jiwer
import jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 46.1 MB/s eta 0:00:00


In [ ]:
def create_recognizer(use_int8):
    suffix = ".int8" if use_int8 else ""

    return sherpa_onnx.OfflineRecognizer.from_whisper(
        encoder=str(
            MODEL_DIR / f"tiny-encoder{suffix}.onnx"
        ),
        decoder=str(
            MODEL_DIR / f"tiny-decoder{suffix}.onnx"
        ),
        tokens=str(MODEL_DIR / "tiny-tokens.txt"),
        language="ta",
        task="transcribe",
        num_threads=2,
        provider="cpu",
        decoding_method="greedy_search",
    )


def evaluate_export(use_int8):
    name = "sherpa_int8" if use_int8 else "sherpa_fp32"
    recognizer = create_recognizer(use_int8)

    rows = []
    chunk_samples = 25 * 16000

    try:
        for index in tqdm(test_indices, desc=name):
            sample = fleurs_test[index]
            audio = sample["audio"]

            waveform = np.asarray(
                audio["array"],
                dtype=np.float32,
            )

            sr = audio["sampling_rate"]

            if sr != 16000:
                waveform = librosa.resample(
                    waveform,
                    orig_sr=sr,
                    target_sr=16000,
                )

            assert len(waveform) > 0, f"Empty audio: {index}"

            parts = []
            start_time = time.perf_counter()

            for start in range(
                0, len(waveform), chunk_samples
            ):
                chunk = np.ascontiguousarray(
                    waveform[start:start + chunk_samples],
                    dtype=np.float32,
                )

                stream = recognizer.create_stream()
                stream.accept_waveform(16000, chunk)
                recognizer.decode_stream(stream)

                parts.append(stream.result.text.strip())
                del stream

            elapsed = time.perf_counter() - start_time

            rows.append({
                "index": index,
                "reference": sample["transcription"],
                "prediction": " ".join(
                    part for part in parts if part
                ),
                "audio_seconds": len(waveform) / 16000,
                "decode_seconds": elapsed,
                "chunks": len(parts),
            })

            if len(rows) % 20 == 0:
                pd.DataFrame(rows).to_csv(
                    EVAL_DIR / f"{name}_predictions.csv",
                    index=False,
                )

        output = pd.DataFrame(rows)

        metrics = {
            "model": name,
            "samples": len(output),
            "wer": 100 * jiwer.wer(
                output["reference"].tolist(),
                output["prediction"].tolist(),
            ),
            "cer": 100 * jiwer.cer(
                output["reference"].tolist(),
                output["prediction"].tolist(),
            ),
            "total_decode_seconds": float(
                output["decode_seconds"].sum()
            ),
            "rtf": float(
                output["decode_seconds"].sum()
                / output["audio_seconds"].sum()
            ),
        }

        output.to_csv(
            EVAL_DIR / f"{name}_predictions.csv",
            index=False,
        )

        (EVAL_DIR / f"{name}_metrics.json").write_text(
            json.dumps(metrics, indent=2)
        )

        return metrics, output

    finally:
        del recognizer
        gc.collect()


fp32_metrics, fp32_output = evaluate_export(False)
int8_metrics, int8_output = evaluate_export(True)

sherpa_fp32:   0%|          | 0/200 [00:00<?, ?it/s]

sherpa_int8:   0%|          | 0/200 [00:00<?, ?it/s]

In [ ]:
comparison = pd.DataFrame([
    fp32_metrics,
    int8_metrics,
]).set_index("model")

display(comparison.round(3))

comparison.to_csv(
    EVAL_DIR / "sherpa_fp32_vs_int8.csv"
)

print(
    "INT8 minus FP32 WER:",
    round(int8_metrics["wer"] - fp32_metrics["wer"], 2),
    "percentage points",
)

print(
    "INT8 minus FP32 CER:",
    round(int8_metrics["cer"] - fp32_metrics["cer"], 2),
    "percentage points",
)

print("Results saved to:", EVAL_DIR)

,samples,wer,cer,total_decode_seconds,rtf
model,,,,,
sherpa_fp32,200,92.803,60.507,607.533,0.233
sherpa_int8,200,96.295,63.427,551.833,0.212


INT8 minus FP32 WER: 3.49 percentage points
INT8 minus FP32 CER: 2.92 percentage points
Results saved to: /content/drive/MyDrive/tamil_whisper_experiment/fleurs_evaluation


### FP32 and INT8 comparison

Both exported models were evaluated on the same 200 FLEURS Tamil recordings with the same decoding setup.

FP32 achieved a WER of 92.80% and CER of 60.51%. INT8 achieved a WER of 96.30% and CER of 63.43%. Quantization therefore increased WER by 3.49 percentage points and CER by 2.92 percentage points.

INT8 completed decoding in 551.83 seconds, compared with 607.53 seconds for FP32—about 9% less time in this run. Both decoded faster than the total audio duration, with real-time factors below 1.

INT8 will be taken forward as the Android deployment candidate, while FP32 is kept as the accuracy reference. The speed gain comes with some loss in recognition quality, and the error rates are still high. Testing on the actual phone is needed before making the final choice.

In [ ]:
# Package INT8 for Android
from pathlib import Path
import shutil
import json

RUN_DIR = Path(
    "/content/drive/MyDrive/tamil_whisper_experiment"
)

SOURCE_DIR = RUN_DIR / "sherpa_onnx"
BUNDLE_DIR = RUN_DIR / "whisper-tamil-int8"
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)

files_to_copy = {
    "tiny-encoder.int8.onnx": "encoder.onnx",
    "tiny-decoder.int8.onnx": "decoder.onnx",
    "tiny-tokens.txt": "tokens.txt",
}

for source_name, destination_name in files_to_copy.items():
    source = SOURCE_DIR / source_name

    assert source.is_file(), f"Missing export: {source}"

    shutil.copy2(
        source,
        BUNDLE_DIR / destination_name,
    )

metadata = {
    "architecture": "multilingual-whisper-tiny",
    "precision": "dynamic-int8",
    "language": "ta",
    "task": "transcribe",
    "evaluation_samples": 200,
    "wer_percent": 96.295,
    "cer_percent": 63.427,
    "status": "Android deployment candidate; not phone-validated",
}

(BUNDLE_DIR / "deployment_info.json").write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8",
)

archive = shutil.make_archive(
    str(RUN_DIR / "whisper-tamil-int8"),
    "zip",
    root_dir=RUN_DIR,
    base_dir=BUNDLE_DIR.name,
)

total_bytes = sum(
    path.stat().st_size
    for path in BUNDLE_DIR.iterdir()
    if path.is_file()
)

print("Bundle:", BUNDLE_DIR)
print(f"Uncompressed bundle size: {total_bytes / 1024**2:.2f} MiB")
print("ZIP saved:", archive)

Bundle: /content/drive/MyDrive/tamil_whisper_experiment/whisper-tamil-int8
Uncompressed bundle size: 98.79 MiB
ZIP saved: /content/drive/MyDrive/tamil_whisper_experiment/whisper-tamil-int8.zip


Saving model and results

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path
import zipfile

RUN_DIR = Path(
    "/content/drive/MyDrive/tamil_whisper_experiment"
)

required_dirs = {
    "huggingface_checkpoint": RUN_DIR / "finetuned",
    "onnx_exports": RUN_DIR / "sherpa_onnx",
    "evaluation": RUN_DIR / "fleurs_evaluation",
}

for label, directory in required_dirs.items():
    assert directory.is_dir(), (
        f"Missing {label}: {directory}"
    )

assert (required_dirs["huggingface_checkpoint"] / "config.json").is_file()

required_exports = [
    "tiny-encoder.onnx",
    "tiny-decoder.onnx",
    "tiny-encoder.int8.onnx",
    "tiny-decoder.int8.onnx",
    "tiny-tokens.txt",
]

for name in required_exports:
    assert (required_dirs["onnx_exports"] / name).is_file(), name

ZIP_PATH = RUN_DIR / "TamilASR_models_and_results.zip"

with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=1,
    allowZip64=True,
) as archive:
    for label, directory in required_dirs.items():
        count = 0

        for path in sorted(directory.rglob("*")):
            if not path.is_file():
                continue

            # The intermediate OpenAI checkpoint is not needed
            # to use the saved HF checkpoint or ONNX exports.
            if label == "onnx_exports" and path.suffix == ".pt":
                continue

            archive.write(
                path,
                arcname=str(
                    Path(label) / path.relative_to(directory)
                ),
            )
            count += 1

        print(f"{label}: packaged {count} files")

assert ZIP_PATH.is_file() and ZIP_PATH.stat().st_size > 0

print(f"Archive size: {ZIP_PATH.stat().st_size / 1024**2:.1f} MiB")
print("Saved to Drive:", ZIP_PATH)

huggingface_checkpoint: packaged 12 files
onnx_exports: packaged 8 files
evaluation: packaged 12 files
Archive size: 353.2 MiB
Saved to Drive: /content/drive/MyDrive/tamil_whisper_experiment/TamilASR_models_and_results.zip
